# Aria Performance Optimization Guide

**Complete guide for profiling, optimization strategies, caching patterns, and performance best practices.**

Learn how to make Aria fast, responsive, and scalable.


## Performance Profiling

### Python Profiling

**Using cProfile for CPU profiling:**

```python
import cProfile
import pstats
from io import StringIO

def profile_function():
    """Profile expensive function."""
    pr = cProfile.Profile()
    pr.enable()

    # Your code here
    expensive_operation()

    pr.disable()
    s = StringIO()
    ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
    ps.print_stats(10)  # Top 10 functions
    print(s.getvalue())

# Usage
profile_function()

# Output shows:
# - ncalls: number of calls
# - tottime: total time in function
# - cumtime: cumulative time in function + called functions
# - filename:lineno(function): location
```

**Using line_profiler for line-by-line:**

```bash
# Install line_profiler
pip install line_profiler

# Profile specific function
# Add @profile decorator to function, then:
kernprof -l -v script.py

# Output shows time per line
Line #  Hits   Time Per Hit % Time Line Contents
  10      1    7.8e+05    7.8e+05    5.2%  result = [i**2 for i in range(1000)]
  11      1    5.4e+05    5.4e+05    3.6%  print(result)
```

**Using memory_profiler for memory usage:**

```bash
pip install memory-profiler

# Profile memory
mprof run script.py
mprof plot mprofile_*.dat

# Check for memory leaks
```

### Database Query Profiling

```python
# Log slow queries
SLOW_QUERY_THRESHOLD = 1.0  # seconds

from flask import g
from time import time

@app.before_request
def start_timer():
    g.start_time = time()

@app.after_request
def log_slow_queries(response):
    elapsed = time() - g.start_time
    if elapsed > SLOW_QUERY_THRESHOLD:
        app.logger.warning(f"Slow query: {elapsed:.2f}s - {request.endpoint}")
    return response

# SQLAlchemy query analysis
from sqlalchemy import event

@event.listens_for(Engine, "before_cursor_execute")
def receive_before_cursor_execute(conn, cursor, statement, parameters, context, executemany):
    conn.info.setdefault('query_start_time', []).append(time())

@event.listens_for(Engine, "after_cursor_execute")
def receive_after_cursor_execute(conn, cursor, statement, parameters, context, executemany):
    total_time = time() - conn.info['query_start_time'].pop(-1)
    if total_time > 0.5:
        print(f"Slow query ({total_time:.2f}s): {statement}")
```

### Application Performance Monitoring (APM)

```python
# Application Insights with custom metrics
from azure.monitor.opentelemetry import configure_azure_monitor

configure_azure_monitor()

# Track custom operation
from opencensus.trace import tracer

with tracer.span(name='expensive_operation') as span:
    span.add_attribute('user_id', current_user.id)
    expensive_operation()
    span.add_attribute('status', 'success')

# View in Azure Portal → Application Insights → Performance
```

---

## Optimization Strategies

### 1. Database Query Optimization

**N+1 Query Problem (BAD):**

```python
# BAD: N+1 queries
users = User.query.all()  # 1 query
for user in users:
    messages = user.messages  # N additional queries!
    print(messages.count())
```

**Eager Loading (GOOD):**

```python
# GOOD: Single query with join
from sqlalchemy.orm import joinedload

users = User.query.options(joinedload(User.messages)).all()
for user in users:
    # messages already loaded
    print(len(user.messages))

# Or use contains_eager for filtered relationships
users = User.query.join(Message).options(
    contains_eager(User.messages)
).filter(Message.created_at > cutoff_date).all()
```

**Query Optimization:**

```python
# Use SELECT * only when needed
# Specify columns instead
messages = db.session.query(
    Message.id,
    Message.content,
    Message.created_at
).filter_by(user_id=user_id).all()

# Use limit
recent_messages = Message.query.order_by(
    Message.created_at.desc()
).limit(10).all()

# Use indexes on frequently filtered columns
class Message(db.Model):
    __table_args__ = (
        db.Index('idx_user_created', 'user_id', 'created_at'),
    )
    id = db.Column(db.Integer, primary_key=True)
    user_id = db.Column(db.Integer)
    created_at = db.Column(db.DateTime)
```

### 2. Caching Strategies

**In-Memory Caching:**

```python
from functools import lru_cache
from functools import wraps
import time

# Simple LRU cache (single process)
@lru_cache(maxsize=128)
def get_user_preferences(user_id: int):
    """Cache user preferences."""
    user = User.query.get(user_id)
    return user.preferences

# Custom cache with TTL
cache_data = {}
cache_timestamps = {}
CACHE_TTL = 300  # 5 minutes

def cache_get(key: str):
    if key in cache_timestamps:
        if time.time() - cache_timestamps[key] > CACHE_TTL:
            del cache_data[key]
            del cache_timestamps[key]
            return None
    return cache_data.get(key)

def cache_set(key: str, value):
    cache_data[key] = value
    cache_timestamps[key] = time.time()

# Usage
def get_chat_history(session_id: str):
    cache_key = f"chat:{session_id}"
    cached = cache_get(cache_key)
    if cached:
        return cached

    history = ChatMessage.query.filter_by(
        session_id=session_id
    ).all()
    cache_set(cache_key, history)
    return history
```

**Redis Caching:**

```python
import redis
import json

redis_client = redis.Redis(host='localhost', port=6379, db=0)

def get_user_settings(user_id: int):
    """Get user settings with Redis cache."""
    cache_key = f"user:{user_id}:settings"

    # Try cache first
    cached = redis_client.get(cache_key)
    if cached:
        return json.loads(cached)

    # Get from database
    user = User.query.get(user_id)
    settings = user.settings.to_dict()

    # Store in cache (expire after 1 hour)
    redis_client.setex(
        cache_key,
        3600,
        json.dumps(settings)
    )

    return settings

# Clear cache on update
def update_user_settings(user_id: int, new_settings: dict):
    user = User.query.get(user_id)
    user.settings = new_settings
    db.session.commit()

    # Invalidate cache
    redis_client.delete(f"user:{user_id}:settings")
```

### 3. Async Operations

**Use async for I/O-bound operations:**

```python
import asyncio
from aiohttp import ClientSession

async def fetch_multiple_apis():
    """Fetch from multiple APIs concurrently."""
    async with ClientSession() as session:
        tasks = [
            fetch_weather_api(session),
            fetch_news_api(session),
            fetch_location_api(session)
        ]
        results = await asyncio.gather(*tasks)
    return results

async def fetch_weather_api(session):
    async with session.get('https://api.weather.com/') as resp:
        return await resp.json()

# Usage in FastAPI
@app.get("/api/data")
async def get_combined_data():
    result = await fetch_multiple_apis()
    return result
```

**Use background tasks for non-critical work:**

```python
from fastapi import BackgroundTasks

@app.post("/api/upload")
async def upload_file(file: UploadFile, background_tasks: BackgroundTasks):
    # Save file immediately
    saved_path = save_file(file)

    # Process in background
    background_tasks.add_task(process_image, saved_path)

    # Return quickly
    return {"status": "uploaded"}

def process_image(path: str):
    """Time-consuming image processing."""
    # Resize, optimize, generate thumbnails, etc.
    pass
```

### 4. Frontend Performance

**Lazy Loading:**

```javascript
// Load images only when visible
const observer = new IntersectionObserver(entries => {
    entries.forEach(entry => {
        if (entry.isIntersecting) {
            const img = entry.target
            img.src = img.dataset.src
            observer.unobserve(img)
        }
    })
})

document.querySelectorAll("img[data-src]").forEach(img => {
    observer.observe(img)
})
```

**Code Splitting:**

```javascript
// Load route components on demand
import React, { lazy, Suspense } from "react"

const ChatPage = lazy(() => import("./ChatPage"))
const Dashboard = lazy(() => import("./Dashboard"))

function App() {
    return (
        <Suspense fallback={<Loading />}>
            <Routes>
                <Route path="/chat" element={<ChatPage />} />
                <Route path="/dashboard" element={<Dashboard />} />
            </Routes>
        </Suspense>
    )
}
```

**Minimize Bundle Size:**

```bash
# Analyze bundle
npm install --save-dev webpack-bundle-analyzer
webpack-bundle-analyzer dist/stats.json

# Remove unused dependencies
npm audit
npm prune

# Use tree-shaking
// Import specific functions
import { debounce } from 'lodash-es';  // GOOD
// Don't import entire library
import _ from 'lodash';  // BAD - includes everything
```

---

## Performance Monitoring

### Metrics to Track

```
1. Response Time
   - Target: <200ms for APIs
   - Monitor: P50, P95, P99

2. Throughput
   - Target: >1000 requests/sec
   - Monitor: requests per second

3. Error Rate
   - Target: <0.1%
   - Monitor: 4xx and 5xx errors

4. Resource Usage
   - CPU: <70%
   - Memory: <80%
   - Disk: <85%

5. Database
   - Query time: P95 <100ms
   - Connection pool: <80% used
   - Replication lag: <1 second
```

### Setting Up Alerts

```bash
# Create alert for slow response times
az monitor metrics alert create \
  --name "Aria-SlowAPI" \
  --resource-group aria-rg \
  --scopes /subscriptions/.../providers/microsoft.insights/components/aria-insights \
  --condition "avg requests/duration > 500" \
  --evaluation-frequency 1m \
  --window-size 5m \
  --severity 2

# Create alert for high error rate
az monitor metrics alert create \
  --name "Aria-HighErrorRate" \
  --resource-group aria-rg \
  --scopes /subscriptions/.../providers/microsoft.insights/components/aria-insights \
  --condition "avg requests/failed > 1%" \
  --evaluation-frequency 1m \
  --window-size 5m \
  --severity 3
```

---

## Performance Checklist

### Before Production

- [ ] Database queries optimized (no N+1)
- [ ] Indexes created on frequently filtered columns
- [ ] Caching strategy implemented
- [ ] Lazy loading enabled for images
- [ ] Code split appropriately
- [ ] Bundle size <500KB
- [ ] Response time <200ms (P95)
- [ ] Error rate <0.5%
- [ ] No memory leaks
- [ ] Connection pools sized correctly

### Post-Deployment

- [ ] Monitoring alerts configured
- [ ] Performance baseline established
- [ ] Real user metrics collected (RUM)
- [ ] Performance regression detected early
- [ ] Cache hit rates >80%
- [ ] Database response times P95 <100ms
- [ ] CPU usage <70% under normal load
- [ ] Memory usage stable


## Performance Optimization Workflow

### Step 1: Identify Bottleneck

```bash
# 1. Check Application Insights
# Look for slow endpoints in Azure Portal

# 2. Run local profiling
kernprof -l -v script.py

# 3. Analyze slow queries
# Check database slow query log
```

### Step 2: Measure Baseline

```python
import time

start = time.perf_counter()
result = expensive_function()
elapsed = time.perf_counter() - start

print(f"Before optimization: {elapsed:.3f}s")
# Before optimization: 5.234s
```

### Step 3: Optimize

```python
# Add caching
@cache_with_ttl(ttl=300)
def expensive_function():
    # Implementation
    pass
```

### Step 4: Verify Improvement

```python
start = time.perf_counter()
result = expensive_function()  # Now cached
elapsed = time.perf_counter() - start

print(f"After optimization: {elapsed:.3f}s")
# After optimization: 0.001s
# Improvement: 5234x faster!
```

### Step 5: Deploy & Monitor

```bash
# Deploy to staging
func azure functionapp publish aria-app-staging

# Run performance tests
python scripts/performance_tests.py

# Monitor in production
# Check Application Insights dashboard
```

---

## Quick Optimization Tips

### Database

```python
# Bad
messages = Message.query.all()
for m in messages:
    print(m.user.name)  # N+1!

# Good
messages = Message.query.options(joinedload(Message.user)).all()
```

### Caching

```python
# Bad
for i in range(100):
    setting = get_setting('TIMEOUT')  # 100 DB queries

# Good
timeout = get_setting('TIMEOUT')  # 1 DB query + 1 cache write
for i in range(100):
    value = timeout  # Use cached value
```

### API Responses

```python
# Bad
return jsonify(user.to_dict())  # Includes all fields

# Good
return jsonify({
    'id': user.id,
    'name': user.name
    # Only necessary fields
})
```

### Async

```python
# Bad
results = []
for url in urls:
    results.append(requests.get(url))  # Sequential

# Good
results = await asyncio.gather(*[
    fetch(url) for url in urls
])  # Concurrent
```
